# Theory

-- QUESTION 1
-- DDL: Defines/modifies database structure.
CREATE TABLE Customers (
    CustomerID INT,
    CustomerName VARCHAR(100)
);

-- DML: Inserts, updates, or deletes data.
INSERT INTO Customers (CustomerID, CustomerName)
VALUES (1, 'John');

-- DQL: Retrieves data.
SELECT * FROM Customers;


-- QUESTION 2
-- SQL constraints maintain data accuracy and integrity.
-- PRIMARY KEY: Uniquely identifies each record.
CustomerID INT PRIMARY KEY;

-- NOT NULL: Prevents NULL values.
CustomerName VARCHAR(100) NOT NULL;

-- FOREIGN KEY: Creates a relationship between tables.
FOREIGN KEY (CustomerID) REFERENCES Customers(CustomerID);


-- QUESTION 3
-- LIMIT = number of records to return.
-- OFFSET = number of records to skip.
-- Page 3 with 10 records per page:
SELECT *
FROM Customers
LIMIT 10 OFFSET 20;


-- QUESTION 4
-- CTE (Common Table Expression) is a temporary named result set.
-- Benefits: improves readability and simplifies complex queries.
WITH HighValueCustomers AS (
    SELECT CustomerID, CustomerName
    FROM Customers
    WHERE TotalSpent > 1000
)
SELECT *
FROM HighValueCustomers;


-- QUESTION 5
-- Normalization organizes data to reduce duplication
-- and improve data integrity.
--
-- 1NF: Atomic values, no repeating groups.
-- 2NF: 1NF + all non-key columns depend on the whole primary key.
-- 3NF: 2NF + non-key columns depend only on the primary key.


-- QUESTION 7
-- Show every customer, including customers with zero orders.
SELECT
    c.CustomerName,
    c.Email,
    COUNT(o.OrderID) AS TotalNumberofOrders
FROM Customers c
LEFT JOIN Orders o
    ON c.CustomerID = o.CustomerID
GROUP BY
    c.CustomerID,
    c.CustomerName,
    c.Email
ORDER BY c.CustomerName;


-- QUESTION 8
-- Display product information with category.
SELECT
    p.ProductName,
    p.Price,
    p.StockQuantity,
    c.CategoryName
FROM Products p
JOIN Categories c
    ON p.CategoryID = c.CategoryID
ORDER BY
    c.CategoryName ASC,
    p.ProductName ASC;


-- QUESTION 9
-- CTE + ROW_NUMBER() to get top 2 expensive products per category.
WITH RankedProducts AS (
    SELECT
        c.CategoryName,
        p.ProductName,
        p.Price,
        ROW_NUMBER() OVER (
            PARTITION BY c.CategoryID
            ORDER BY p.Price DESC
        ) AS RankNumber
    FROM Products p
    JOIN Categories c
        ON p.CategoryID = c.CategoryID
)
SELECT
    CategoryName,
    ProductName,
    Price
FROM RankedProducts
WHERE RankNumber <= 2
ORDER BY CategoryName, Price DESC;


-- QUESTION 10.1
-- Top 5 customers based on total amount spent.
SELECT
    CONCAT(c.first_name, ' ', c.last_name) AS CustomerName,
    c.email,
    SUM(p.amount) AS TotalAmountSpent
FROM customer c
JOIN payment p
    ON c.customer_id = p.customer_id
GROUP BY
    c.customer_id,
    c.first_name,
    c.last_name,
    c.email
ORDER BY TotalAmountSpent DESC
LIMIT 5;


-- QUESTION 10.2
-- Top 3 movie categories based on rental count.
SELECT
    cat.name AS CategoryName,
    COUNT(r.rental_id) AS RentalCount
FROM category cat
JOIN film_category fc
    ON cat.category_id = fc.category_id
JOIN inventory i
    ON fc.film_id = i.film_id
JOIN rental r
    ON i.inventory_id = r.inventory_id
GROUP BY
    cat.category_id,
    cat.name
ORDER BY RentalCount DESC
LIMIT 3;


-- QUESTION 10.3
-- Total films available at each store
-- and films that have never been rented.
SELECT
    i.store_id AS StoreID,
    COUNT(i.inventory_id) AS TotalFilms,
    SUM(
        CASE
            WHEN r.rental_id IS NULL THEN 1
            ELSE 0
        END
    ) AS NeverRented
FROM inventory i
LEFT JOIN rental r
    ON i.inventory_id = r.inventory_id
GROUP BY i.store_id
ORDER BY i.store_id;


-- QUESTION 10.4
-- Total revenue per month for 2023.
SELECT
    MONTH(payment_date) AS MonthNumber,
    SUM(amount) AS TotalRevenue
FROM payment
WHERE payment_date >= '2023-01-01'
  AND payment_date < '2024-01-01'
GROUP BY MONTH(payment_date)
ORDER BY MonthNumber;


-- QUESTION 10.5
-- Customers who rented more than 10 times
-- in the last 6 months.
SELECT
    c.customer_id,
    CONCAT(c.first_name, ' ', c.last_name) AS CustomerName,
    COUNT(r.rental_id) AS RentalCount
FROM customer c
JOIN rental r
    ON c.customer_id = r.customer_id
WHERE r.rental_date >= DATE_SUB(CURDATE(), INTERVAL 6 MONTH)
GROUP BY
    c.customer_id,
    c.first_name,
    c.last_name
HAVING COUNT(r.rental_id) > 10
ORDER BY RentalCount DESC;